# 02 — Cross-validation design and baseline model

How the accounts are split into 5 folds so that no laundering operation, and where possible no owner, is on both sides; the positive rate is equal across folds. In round k: test = fold k, val = fold k+1 (early stopping and threshold only), train = the other three. Every account gets one score from a model that never saw it. Code: `src/skeptical_second_reader/split.py`; leakage tests: `tests/test_split.py`. Aggregates only.

## 0. Setup

In [1]:
import hashlib

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_key,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.split import (
    N_FOLDS,
    PATTERN_PREFIX,
    SEED,
    assign_folds,
    fold_roles,
    split_groups,
)

In [2]:
%%time
df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
members = pattern_members(read_patterns(PATTERNS_TXT))
assert members["account"].isin(labels.index).all(), "pattern account not found"
n_pattern_accounts = members["account"].nunique()
n_pos, n_owners = labels.sum(), owners.nunique()
print(f"accounts: {labels.size:,}  positives: {n_pos:,}  owners: {n_owners:,}")
n_attempts = members["attempt"].nunique()
print(f"attempts: {n_attempts}  pattern accounts: {n_pattern_accounts:,}")
print(f"folds: {N_FOLDS}  seed: {SEED}")

accounts: 515,088  positives: 6,357  owners: 166,207
attempts: 370  pattern accounts: 3,170
folds: 5  seed: 42
CPU times: total: 14.6 s
Wall time: 12.1 s


## 1. Option A — keep each owner together

Does the owner split of notebook 01 also keep each laundering pattern in one fold? If not, the model trains on one part of an operation and is tested on another.

In [3]:
folds_a = assign_folds(labels, owners)
assert (folds_a.groupby(owners).nunique() == 1).all()

per_attempt = (
    members.assign(
        owner=members["account"].map(owners), fold=members["account"].map(folds_a)
    )
    .groupby("attempt")
    .agg(owners=("owner", "nunique"), folds=("fold", "nunique"))
)
spread_a = (per_attempt["folds"] > 1).sum()
print(f"attempts with a single owner: {(per_attempt['owners'] == 1).sum()}")
med, mx = per_attempt["owners"].median(), per_attempt["owners"].max()
print(f"owners per attempt: median {med:.0f}, max {mx}")
n_att = len(per_attempt)
print(f"attempts spread over 2+ folds: {spread_a} of {n_att} ({spread_a / n_att:.1%})")
share = n_pattern_accounts / labels.sum()
print(f"pattern accounts: {n_pattern_accounts:,} = {share:.1%} of all positives")

# Time span of one attempt: first to last transaction, in days.
times = read_patterns(PATTERNS_TXT).groupby("attempt")["timestamp"]
span_days = (times.max() - times.min()).dt.total_seconds() / 86400
print(
    f"attempt span in days: median {span_days.median():.1f}, max {span_days.max():.1f}"
)

attempts with a single owner: 0
owners per attempt: median 8, max 45
attempts spread over 2+ folds: 365 of 370 (98.6%)
pattern accounts: 3,170 = 49.9% of all positives
attempt span in days: median 3.1, max 8.4


**Result:** No. A pattern uses accounts of a median of 8 owners (up to 45), so 365 of the 370 patterns are spread over folds (332–364 when only the group names change). A pattern spans a median of 3.1 days (at most 8.4), and pattern accounts are half of the positives (49.9%), so option A would score optimistically.

## 2. Option B — keep owners and patterns together

Link accounts that share an owner or a pattern and keep each connected group in one fold. This works only if no group is too large: a fold holds 20% of the accounts and should hold about 20% of the positives.

In [4]:
# Graph: account -- owner and account -- attempt edges.
n_acc = labels.size
owner_code, owner_ids = pd.factorize(owners)
attempt_ids = pd.Index(members["attempt"].unique())
position = pd.Series(np.arange(n_acc), index=labels.index)
rows = np.concatenate([np.arange(n_acc), position[members["account"]].to_numpy()])
cols = np.concatenate(
    [
        n_acc + owner_code,
        n_acc + owner_ids.size + attempt_ids.get_indexer(members["attempt"]),
    ]
)
n_nodes = n_acc + owner_ids.size + attempt_ids.size
graph = coo_matrix((np.ones(rows.size), (rows, cols)), shape=(n_nodes, n_nodes))
_, node_component = connected_components(graph, directed=False)
component = pd.Series(node_component[:n_acc], index=labels.index)

sizes = component.value_counts()
in_largest = component == sizes.index[0]
pos_largest = labels[in_largest].sum()
print(f"components: {sizes.size:,}")
print(f"largest: {sizes.iloc[0]:,} accounts ({in_largest.mean():.1%} of accounts)")
print(f"positives in largest: {pos_largest:,} ({pos_largest / n_pos:.1%})")
print("five largest components:", sizes.head(5).tolist())

components: 163,414


largest: 99,978 accounts (19.4% of accounts)
positives in largest: 3,259 (51.3%)
five largest components: [99978, 2511, 2113, 928, 703]


**Result:** Not possible. Owners link patterns into one group of 99,978 accounts (19.4%) holding 3,259 positives (51.3%); its fold would hold half of the positives.

## 3. Option D (chosen) — pattern first, then owner

Keep each pattern together (patterns that share an account merged), and owners together only outside patterns. Checks: (1) no pattern crosses folds, (2) the folds stay balanced, (3) how many owners are split, (4) train, val and test share no group, (5) how far transactions alone connect accounts, (6) how much the counts depend on one fold assignment.

In [5]:
groups = split_groups(owners, members)
folds = assign_folds(labels, groups)

# Check 1: no pattern crosses folds; groups stay in one fold.
assert (folds.groupby(groups).nunique() == 1).all(), "a group is in two folds"
spread_d = (
    members.assign(fold=members["account"].map(folds))
    .groupby("attempt")["fold"]
    .nunique()
    > 1
).sum()
assert spread_d == 0, "a pattern crosses folds"
pattern_groups = groups[groups.str.startswith(PATTERN_PREFIX)]
largest_pattern = pattern_groups.value_counts().iloc[0]
print(
    f"pattern groups: {pattern_groups.nunique()}  largest: {largest_pattern} accounts"
)
print(f"attempts spread over 2+ folds: {spread_d}")

# Check 3: owners that are now split, and what they hold.
split_owner_ids = folds.groupby(owners).nunique().loc[lambda s: s > 1].index
in_split = owners.isin(split_owner_ids)
print(f"owners split across folds: {split_owner_ids.size:,} of {owners.nunique():,}")
print(f"accounts of split owners: {in_split.sum():,} ({in_split.mean():.1%})")
print(f"positives among them: {labels[in_split].sum():,}")

# Check 2: fold balance.
per_fold = pd.DataFrame(
    {
        "accounts": folds.value_counts().sort_index(),
        "groups": groups.groupby(folds).nunique(),
        "positives": labels.groupby(folds).sum(),
    }
)
per_fold["positive_rate"] = (per_fold["positives"] / per_fold["accounts"]).round(4)
per_fold

pattern groups: 210  largest: 330 accounts
attempts spread over 2+ folds: 0


owners split across folds: 1,532 of 166,207
accounts of split owners: 89,935 (17.5%)
positives among them: 2,252


,accounts,groups,positives,positive_rate
fold,,,,
0,103018,33089,1271,0.0123
1,103018,33066,1272,0.0123
2,103017,33094,1271,0.0123
3,103017,33089,1271,0.0123
4,103018,33113,1272,0.0123


In [6]:
# Check 4: roles in every round are disjoint, cover all accounts, share no group.
rows = []
for k in range(N_FOLDS):
    roles = fold_roles(folds, k)
    assert (roles.groupby(groups).nunique() == 1).all(), f"round {k}: group split"
    counts = roles.value_counts()
    assert counts.sum() == labels.size
    pos = labels.groupby(roles).sum()
    rows.append({"round": k, **counts.to_dict(), **pos.add_prefix("pos_").to_dict()})
pd.DataFrame(rows).set_index("round")[
    ["train", "val", "test", "pos_train", "pos_val", "pos_test"]
]

,train,val,test,pos_train,pos_val,pos_test
round,,,,,,
0,309052,103018,103018,3814,1272,1271
1,309053,103017,103018,3814,1271,1272
2,309054,103017,103017,3815,1271,1271
3,309053,103018,103017,3814,1272,1271
4,309052,103018,103018,3814,1271,1272


In [7]:
# Check 5: how far do transactions alone connect the accounts?
senders = labels.index.get_indexer(account_key(df["from_bank"], df["from_account"]))
receivers = labels.index.get_indexer(account_key(df["to_bank"], df["to_account"]))
assert (senders >= 0).all() and (receivers >= 0).all(), "transaction account not found"
tx_graph = coo_matrix(
    (np.ones(senders.size), (senders, receivers)), shape=(labels.size, labels.size)
)
_, tx_component = connected_components(tx_graph, directed=False)
tx_sizes = np.bincount(tx_component)
print(f"transaction components: {tx_sizes.size:,}")
largest_tx = tx_sizes.max()
print(f"largest: {largest_tx:,} accounts ({largest_tx / labels.size:.1%} of accounts)")

transaction components: 114,139
largest: 372,089 accounts (72.2% of accounts)


In [8]:
# Check 6: how much do the reported counts depend on this one fold assignment?
# Same groups and seed, only the group names change. The folds used by the model
# stay the ones above.
def random_names(values, salt):
    names = values.astype(str)
    unique = names.unique()
    new = {u: hashlib.md5(f"{salt}:{u}".encode()).hexdigest() for u in unique}
    return names.map(new)


def patterns_crossing(fold_of):
    fold_per_member = members["account"].map(fold_of)
    return (fold_per_member.groupby(members["attempt"]).nunique() > 1).sum()


rows = []
for salt in range(5):
    folds_a_i = assign_folds(labels, random_names(owners, salt))
    folds_d_i = assign_folds(labels, random_names(groups, salt))
    per_owner = folds_d_i.groupby(owners).nunique()
    inside = owners.isin(per_owner[per_owner > 1].index)
    pos_per_fold = labels.groupby(folds_d_i).sum()
    rows.append(
        {
            "naming": salt,
            "A_patterns_crossing": patterns_crossing(folds_a_i),
            "D_patterns_crossing": patterns_crossing(folds_d_i),
            "D_split_owners": per_owner.gt(1).sum(),
            "D_split_accounts_pct": round(100 * inside.mean(), 2),
            "D_split_positives": labels[inside].sum(),
            "D_fold_pos_min": pos_per_fold.min(),
            "D_fold_pos_max": pos_per_fold.max(),
        }
    )
pd.DataFrame(rows).set_index("naming")

,A_patterns_crossing,D_patterns_crossing,D_split_owners,D_split_accounts_pct,D_split_positives,D_fold_pos_min,D_fold_pos_max
naming,,,,,,,
0,364,0,1513,17.68,2244,1271,1272
1,358,0,1547,17.89,2276,1271,1272
2,352,0,1533,17.41,2264,1271,1272
3,334,0,1518,17.58,2244,1271,1272
4,332,0,1537,17.64,2261,1271,1272


**Result:** Patterns form 210 groups (largest 330 accounts) and none crosses folds. Each fold holds 103,017–103,018 accounts and 1,271–1,272 positives (1.23%); train, val and test share no group in any round. The price: 1,532 owners are split, holding 89,935 accounts (17.5%) and 2,252 positives (1,513–1,547 owners when only the group names change). We accept it because pattern accounts share laundering transactions and are all positive, while positives do not cluster inside owners (notebook 01), and a bank sees its owners again, so their overlap is close to real use (Kapoor & Narayanan, 2022, p. 5); a new laundering operation is by definition unseen. Transactions alone connect 72.2% of accounts, so no split can separate every link.

## References

Kapoor, S., Narayanan, A. (2022). Leakage and the reproducibility crisis in ML-based science. *arXiv:2207.07048*. https://arxiv.org/abs/2207.07048